# Cluster Forecasting Experiment Runner

This notebook keeps the forecasting features built from raw 2023 daily usage data.
Cluster labels are only used to route households into cluster-specific models.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_NAME = "Clustering-And-Forecasting-TimeSeries-PlayingGround"
REPO_URL = "https://github.com/QuirkyCroissant/Clustering-And-Forecasting-TimeSeries-PlayingGround.git"
BRANCH = os.environ.get("NOTEBOOK_GIT_BRANCH", "feat/cluster-forecasting-lgbm-xgb")

kaggle_root = Path("/kaggle/working")
cwd = Path.cwd().resolve()

if cwd.name == REPO_NAME and (cwd / "src" / "forecasting").exists():
    REPO_ROOT = cwd
elif kaggle_root.exists():
    repo_dir = kaggle_root / REPO_NAME
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)
    REPO_ROOT = repo_dir.resolve()
else:
    raise RuntimeError(
        "Could not determine REPO_ROOT automatically. Open the notebook from the repo root or run it on Kaggle."
    )

SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_PATH:", SRC_PATH)

In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

from forecasting.data import (
    discover_cluster_cases,
    ensure_experiment_dirs,
    ensure_output_dirs,
    load_wide_csv,
)

OUT = ensure_output_dirs(REPO_ROOT)

TRAIN_23_PATH = REPO_ROOT / "data" / "raw" / "sample_23.csv"
TEST_24_PATH = REPO_ROOT / "data" / "raw" / "sample_24.csv"
CLUSTER_CASE_DIR = REPO_ROOT / "notebooks" / "outputs" / "feature"
SHAPELET_STATIC_PATH = REPO_ROOT / "notebooks" / "outputs" / "shapelet_experiments" / "shapelet_features.csv"

In [ ]:
DEBUG = False
DEBUG_FRAC = 0.2

MODEL_NAME = "xgb"
GPU_ENABLED = True
MIN_CLUSTER_ROWS = 500
MIN_CLUSTER_HOUSEHOLDS = 30

SELECTED_CASES = None
INCLUDE_BASE_VARIANT = True
INCLUDE_SHAPELET_STATIC_VARIANT = True

PLOT_SAMPLE_PER_GROUP = 2
PLOT_MAX_GROUPS = None
RANDOM_STATE = 42

In [ ]:
def build_experiment_configs(case_paths: dict[str, Path]) -> list[dict]:
    case_names = SELECTED_CASES or list(case_paths)
    missing = sorted(set(case_names) - set(case_paths))
    if missing:
        raise ValueError(f"Unknown case names requested: {missing}")

    variant_defs = []
    if INCLUDE_BASE_VARIANT:
        variant_defs.append(("base", None))
    if INCLUDE_SHAPELET_STATIC_VARIANT:
        if not SHAPELET_STATIC_PATH.exists():
            raise FileNotFoundError(f"Missing shapelet static feature file: {SHAPELET_STATIC_PATH}")
        variant_defs.append(("shapelet_static", SHAPELET_STATIC_PATH))

    configs = []
    for case_name in case_names:
        for variant_name, static_path in variant_defs:
            configs.append(
                {
                    "case_name": case_name,
                    "variant_name": variant_name,
                    "experiment_name": f"{case_name}_{variant_name}",
                    "cluster_path": str(case_paths[case_name]),
                    "static_features_path": str(static_path) if static_path else None,
                }
            )
    return configs


def build_run_settings() -> dict:
    return {
        "debug": DEBUG,
        "debug_frac": DEBUG_FRAC,
        "model_name": MODEL_NAME,
        "gpu_enabled": GPU_ENABLED,
        "min_cluster_rows": MIN_CLUSTER_ROWS,
        "min_cluster_households": MIN_CLUSTER_HOUSEHOLDS,
        "plot_sample_per_group": PLOT_SAMPLE_PER_GROUP,
        "plot_max_groups": PLOT_MAX_GROUPS,
        "random_state": RANDOM_STATE,
    }


def run_experiment_subprocess(exp_config: dict, run_settings: dict):
    env = os.environ.copy()
    existing_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = str(SRC_PATH) if not existing_pythonpath else str(SRC_PATH) + os.pathsep + existing_pythonpath

    cmd = [
        sys.executable,
        "-m",
        "forecasting.experiment",
        "--repo-root",
        str(REPO_ROOT),
        "--config-json",
        json.dumps(exp_config),
        "--settings-json",
        json.dumps(run_settings),
    ]

    try:
        subprocess.run(cmd, check=True, cwd=REPO_ROOT, env=env)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f"Experiment failed: {exp_config['experiment_name']}") from exc


def load_experiment_metric(exp_name: str, table_name: str) -> pd.DataFrame:
    metric_path = ensure_experiment_dirs(REPO_ROOT, exp_name)["metrics"] / f"{table_name}.csv"
    if not metric_path.exists():
        raise FileNotFoundError(f"Missing metric output for {exp_name}: {metric_path}")
    return pd.read_csv(metric_path)

In [ ]:
cluster_case_paths = discover_cluster_cases(CLUSTER_CASE_DIR)
experiment_configs = build_experiment_configs(cluster_case_paths)
run_settings = build_run_settings()

experiment_plan = pd.DataFrame(experiment_configs)
display(experiment_plan)
print(f"Planned experiments: {len(experiment_configs)}")

In [ ]:
completed_experiments = []

for exp_config in experiment_configs:
    print(f"\n=== Running {exp_config['experiment_name']} ===")
    run_experiment_subprocess(exp_config, run_settings)
    completed_experiments.append(exp_config["experiment_name"])

In [ ]:
all_overall_summary = pd.concat(
    [load_experiment_metric(exp_name, "overall_summary") for exp_name in completed_experiments],
    ignore_index=True,
)
all_route_summary = pd.concat(
    [load_experiment_metric(exp_name, "route_summary") for exp_name in completed_experiments],
    ignore_index=True,
)
all_cluster_mae_summary = pd.concat(
    [load_experiment_metric(exp_name, "cluster_mae_summary") for exp_name in completed_experiments],
    ignore_index=True,
)
all_cluster_compare_summary = pd.concat(
    [load_experiment_metric(exp_name, "cluster_compare_summary") for exp_name in completed_experiments],
    ignore_index=True,
)

model_comparison = (
    all_overall_summary
    .pivot(index="experiment_name", columns="model", values="mean_mae")
    .reset_index()
)
model_comparison["delta_cluster_minus_global"] = (
    model_comparison[f"cluster_{MODEL_NAME}"] - model_comparison[f"global_{MODEL_NAME}"]
)
model_comparison = model_comparison.sort_values("delta_cluster_minus_global").reset_index(drop=True)

all_overall_summary.to_csv(OUT["metrics"] / "all_experiments_overall_summary.csv", index=False)
all_route_summary.to_csv(OUT["metrics"] / "all_experiments_route_summary.csv", index=False)
all_cluster_mae_summary.to_csv(OUT["metrics"] / "all_experiments_cluster_mae_summary.csv", index=False)
all_cluster_compare_summary.to_csv(OUT["metrics"] / "all_experiments_cluster_compare_summary.csv", index=False)
model_comparison.to_csv(OUT["metrics"] / "all_experiments_model_comparison.csv", index=False)

display(all_overall_summary.sort_values(["experiment_name", "model"]).reset_index(drop=True))
display(model_comparison)
display(all_cluster_compare_summary.head(20))